Links: [Reflexion: Language Agents with Verbal Reinforcement Learning
](https://arxiv.org/abs/2303.11366), [Github](https://github.com/noahshinn/reflexion/blob/main/programming_runs/immediate_reflexion.py), [Blog](https://nanothoughts.substack.com/p/reflecting-on-reflexion)

### Movation
- Human can learn from trial-and-error and how to formalize this concept for agent?
- While RL usually uses a sparse scalar reward to train agent, can agent use natural langauge critiques to learn and improve?

### Problem
Training LLM agents with RL suffer from sample inefficiency and optimizeation difficulty (e.g.. define a good reward function)

### Key Insights
- Verbal Reinforcement: Instead of updating model weights via scalar rewards, Reflexion updates the agent's context window with verbal feedback. The "gradient" is a natural language summary of why a previous attempt failed, providing a concrete direction for improvement in the next episode.
- Episodic Memory: The policy $\pi_\theta$ is parameterized by the fixed LLM weights $\theta$ and a dynamic memory component $mem$. Optimizing the choice of action becomes a function of optimizing the verbal context stored in $mem$.
- Self-Reflection >> Self-Refinement: "Self-Refine" methods operate on a single generation, while self-reflection stores all the lessons across trials

### Method (Pesudo Code)

INIT:
- $M_e$: evaluator model for $M_{a}$'s rollout trajectory
- $M_{sr}$: self-reflection model to provide reflections
- $M_{a}$: actor model to sample rollout trajectories

Initialize policy $\pi_{\theta}(a_i \mid s_i)$, where $\theta = ({M_a, mem})$ and $mem$ is empty memory

Generate intial trajectory $\tau_0$ by sampling from $\pi_{\theta}$

Evaluate $\tau_0$ with $M_e$

Generate the first reflection $sr_0$ using $M_{sr}$ on $\tau_0$

Append to memory $mem \leftarrow sr_0$

Set $t=0$

while $M_e$ not pass or $t < \text{max trials}$ do
- Step 1: Generate $\tau_t = [a_0, o_0, ..., a_i, o_i]$ with $\pi_{\theta}$
- Step 2: Evaluate $\tau_t$ with $M_e$
- Step 3: Generate reflection $sr_t$ using $M_{sr}$ on $\tau_t$
- Step 4: Append $sr_t$ to $mem$
- Increment t

end while

return

### Reflection
- There are some parallels between language-based vs gradient-based update algorithms
    * **Gradient Descent**: (prediction, ground truth) $\rightarrow$ loss $\rightarrow$ gradient $\rightarrow$ update with optimizer
    * **Natural Language Critique**: (prediction, ground truth) $\rightarrow$ critques (if it is wrong, how it is wrong) $\rightarrow$ reflection (how to improve, what should keep in mind, etc.) $\rightarrow$ update by putting reflection + prior failed attempts in context
- Neither AlphaEvolve nor Reflexion includes any model weight updates $\rightarrow$ Can we combine natural language critiques + reflection and gradient-based method. I am thinking about something like context distillation
- This is very similar to the [AlphaEvolve](https://deepmind.google/blog/alphaevolve-a-gemini-powered-coding-agent-for-designing-advanced-algorithms/) idea, which comes later in time and its variants, which leverage test-time compute to refine its trajectory and answers given natural language critiques or scalar rewards for next iteration
- This also reminds me of sequential/parallel/hybrid branching at test-time (e.g., [Learning Adaptive Parallel Reasoning with Language Models
](https://arxiv.org/abs/2504.15466)). Here, all the attempts are conditioned on the prior history and can be thoughts as sequentialize parallel trials. 
- The paper is also very well-written. Reflexion is proposed as a framework to perform test-time improvement through iterative natural langauge critiques and reflections. The implementation or $M_{e}$ $M_{sr}$ is flexible: HotpotQA just uses natural langauge, HumanEvals uses unit tests for evaluators